# Transportability Quickstart
An effect estimated in a **source** population may not hold in a **target** population
if the two differ in effect modifiers. This notebook shows the problem, a doubly robust
correction, and the overlap diagnostic that tells you whether correction is trustworthy.

Setup: both populations share the same structural model, but the effect of treatment
depends on `X` and `X` is distributed differently in the two populations. We observe
treatment and outcome in the source, and **only covariates** in the target.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LogisticRegression

PS_CLIP = 0.01

## 1. Source and target populations
Structural model (same in both): `Y = 2X + 1.5A + 0.5·A·X + noise`, so the conditional
effect is `1.5 + 0.5·X` and **grows linearly with X**. The source has `X ~ N(0, 1.5²)`
and the target has `X ~ N(1, 1²)`.

The source is deliberately the *wider* distribution, so every target covariate value
has source counterparts. (An earlier draft used a target wider than the source: the
density ratio is then unbounded and transported estimates became several times noisier.
That is the poor-overlap case, and section 5 shows how to spot it.)

In [2]:
def make_population(n, x_mean, x_sd, seed):
    r = np.random.default_rng(seed)
    X = r.normal(x_mean, x_sd, n)
    A = r.binomial(1, 1 / (1 + np.exp(-0.8 * X)))
    Y = 2.0 * X + 1.5 * A + 0.5 * A * X + r.normal(0, 1, n)
    return pd.DataFrame({"X": X, "A": A, "Y": Y})

def true_ate_for(x):
    # E[Y(1) - Y(0) | X = x] = 1.5 + 0.5 x, averaged over the covariates
    return (1.5 + 0.5 * np.asarray(x)).mean()

SRC = dict(x_mean=0.0, x_sd=1.5)
TGT = dict(x_mean=1.0, x_sd=1.0)
source = make_population(3000, seed=1, **SRC)
target_X = make_population(1000, seed=2, **TGT).X.values   # A and Y discarded: unavailable in practice

print(f"Source: mean X = {source.X.mean():.3f}, sd = {source.X.std():.3f}")
print(f"Target: mean X = {target_X.mean():.3f}, sd = {target_X.std():.3f}")
print(f"True ATE in source: {true_ate_for(source.X):.3f}")
print(f"True ATE in target: {true_ate_for(target_X):.3f}")

Source: mean X = -0.004, sd = 1.494
Target: mean X = 0.978, sd = 1.013
True ATE in source: 1.498
True ATE in target: 1.989


## 2. Estimators
- `transported_ate(form="dr")`: **doubly robust transport estimator.** Evaluates the
  outcome-model effect `m1(X) - m0(X)` on the *target* covariates, then adds a
  source-residual correction weighted by the odds of being in the target.
  It stays consistent if either the outcome model or the selection model is right.
- `transported_ate(form="weighted_aipw")`: the form from the first draft. It takes the
  source AIPW scores and reweights them toward the target. It never evaluates the outcome
  model on the target, so it relies on the selection model being right.
- The selection model estimates `P(target | X)` from the stacked covariates. For Gaussian
  covariates with different variances the log odds are quadratic in `X`, so
  `quad_selection=True` adds an `X²` term.

In [3]:
def _features(x, quad):
    x = np.asarray(x).reshape(-1, 1)
    return np.hstack([x, x ** 2]) if quad else x

def selection_odds(Xs, Xt, quad):
    # P(target | X) / P(source | X) for each source row; equals (n_t / n_s) * f_t(x) / f_s(x)
    Z = np.vstack([_features(Xs, quad), _features(Xt, quad)])
    S = np.r_[np.zeros(len(Xs)), np.ones(len(Xt))]
    p = LogisticRegression(C=1e6, max_iter=5000).fit(Z, S).predict_proba(_features(Xs, quad))[:, 1]
    return p / (1 - p)

def _nuisances(source, outcome):
    Xs, A, Y = source[["X"]].values, source["A"].values, source["Y"].values
    ps = LogisticRegression(C=1e6, max_iter=5000).fit(Xs, A).predict_proba(Xs)[:, 1]
    ps = np.clip(ps, PS_CLIP, 1 - PS_CLIP)
    if outcome == "correct":
        f1 = LinearRegression().fit(Xs[A == 1], Y[A == 1]).predict
        f0 = LinearRegression().fit(Xs[A == 0], Y[A == 0]).predict
    else:                                    # deliberately wrong: ignores X
        c1, c0 = Y[A == 1].mean(), Y[A == 0].mean()
        f1 = lambda x: np.full(len(x), c1)
        f0 = lambda x: np.full(len(x), c0)
    return ps, f1, f0

def source_ate(source, outcome="correct"):
    Xs, A, Y = source[["X"]].values, source["A"].values, source["Y"].values
    ps, f1, f0 = _nuisances(source, outcome)
    m1, m0 = f1(Xs), f0(Xs)
    return ((m1 - m0) + A * (Y - m1) / ps - (1 - A) * (Y - m0) / (1 - ps)).mean()

def transported_ate(source, target_X, form="dr", quad_selection=True, outcome="correct"):
    Xs, A, Y = source[["X"]].values, source["A"].values, source["Y"].values
    Xt = np.asarray(target_X).reshape(-1, 1)
    ps, f1, f0 = _nuisances(source, outcome)
    m1, m0 = f1(Xs), f0(Xs)
    resid = A * (Y - m1) / ps - (1 - A) * (Y - m0) / (1 - ps)
    w = selection_odds(Xs, Xt, quad_selection)
    if form == "dr":
        return (f1(Xt) - f0(Xt)).mean() + (w * resid).sum() / len(Xt)
    return np.average((m1 - m0) + resid, weights=w)

## 3. The problem: what does the source tell us about the target?
Two separate things go wrong if you carry a source number to the target, and it helps to
keep them apart:
- **Confounding** (treatment depends on `X`): the naive difference in means is biased even
  *within* the source.
- **Transport shift** (`X` is distributed differently): even a perfectly confounding-adjusted
  source estimate answers the source question, not the target question.

The source-adjusted AIPW estimate removes the first problem, so its gap to the target truth
isolates the second.

In [4]:
naive = source[source.A == 1].Y.mean() - source[source.A == 0].Y.mean()
adj = source_ate(source)
truth_s, truth_t = true_ate_for(source.X), true_ate_for(target_X)

print(f"Naive source diff-in-means:   {naive:6.3f}   (truth in source {truth_s:.3f}: confounding bias {naive - truth_s:+.3f})")
print(f"Source-adjusted AIPW:         {adj:6.3f}   (truth in source {truth_s:.3f})")
print(f"Target true ATE:              {truth_t:6.3f}")
print(f"-> adjusted source estimate is off by {adj - truth_t:+.3f} for the target: that is the transport shift.")

Naive source diff-in-means:    4.671   (truth in source 1.498: confounding bias +3.173)
Source-adjusted AIPW:          1.530   (truth in source 1.498)
Target true ATE:               1.989
-> adjusted source estimate is off by -0.459 for the target: that is the transport shift.


## 4. The correction
Both transport forms below use a quadratic selection model (correct here) and a correct
outcome model, so both should land near the target truth. The Monte Carlo cell afterwards
shows where they differ.

In [5]:
print(f"Target true ATE:                         {truth_t:.3f}")
print(f"Source-adjusted AIPW (no transport):     {adj:.3f}")
print(f"Transported, DR form, quad selection:    {transported_ate(source, target_X, 'dr', True):.3f}")
print(f"Transported, weighted-AIPW, quad sel.:   {transported_ate(source, target_X, 'weighted_aipw', True):.3f}")

Target true ATE:                         1.989
Source-adjusted AIPW (no transport):     1.530
Transported, DR form, quad selection:    1.986
Transported, weighted-AIPW, quad sel.:   1.988


## 5. Overlap diagnostic
Correction needs overlap: target covariate values must also occur in the source. The weights
that move the source toward the target show how much the estimate leans on a few rows.

- **Effective sample size (ESS)** of the weights, `(Σw)² / Σw²`, compared with the source
  sample size. A small ratio means a few rows dominate.
- **Largest single weight** as a share of the total.
- **Covariate ranges**: central 98% of the target against the source.

In [6]:
Xs = source.X.values
w = selection_odds(Xs, target_X, quad=True)
ess = w.sum() ** 2 / (w ** 2).sum()
print(f"Source rows: {len(Xs)}   Effective sample size of transport weights: {ess:.0f} ({ess / len(Xs):.0%})")
print(f"Largest single weight: {w.max() / w.sum():.2%} of total weight")
print(f"Source X, central 98%: [{np.percentile(Xs, 1):.2f}, {np.percentile(Xs, 99):.2f}]")
print(f"Target X, central 98%: [{np.percentile(target_X, 1):.2f}, {np.percentile(target_X, 99):.2f}]")
print("-> A low ESS ratio, one dominant weight, or target values outside the source range")
print("   mean the estimate is extrapolating: report it with caution or do not transport.")

Source rows: 3000   Effective sample size of transport weights: 1904 (63%)
Largest single weight: 0.07% of total weight
Source X, central 98%: [-3.63, 3.47]
Target X, central 98%: [-1.28, 3.62]
-> A low ESS ratio, one dominant weight, or target values outside the source range
   mean the estimate is extrapolating: report it with caution or do not transport.


## 6. Does it work on average? Monte Carlo
One draw proves little. This repeats the whole simulation 100 times and reports the mean,
bias against the *replication's own* target truth, and the spread. The last rows
deliberately break one model at a time to show what "doubly robust" buys.

In [7]:
R = 100
configs = {
    "source-adjusted, no transport":                   lambda s, t: source_ate(s),
    "DR | selection right, outcome right":             lambda s, t: transported_ate(s, t, "dr", True, "correct"),
    "DR | selection WRONG (linear), outcome right":    lambda s, t: transported_ate(s, t, "dr", False, "correct"),
    "DR | selection right, outcome WRONG":             lambda s, t: transported_ate(s, t, "dr", True, "wrong"),
    "weighted-AIPW | selection WRONG, outcome right":  lambda s, t: transported_ate(s, t, "weighted_aipw", False, "correct"),
    "weighted-AIPW | selection right, outcome right":  lambda s, t: transported_ate(s, t, "weighted_aipw", True, "correct"),
}
est = {k: [] for k in configs}
truths = []
for r in range(R):
    s = make_population(3000, seed=1000 + r, **SRC)
    t = make_population(1000, seed=5000 + r, **TGT).X.values
    truths.append(true_ate_for(t))
    for k, f in configs.items():
        est[k].append(f(s, t))
truths = np.array(truths)
print(f"{R} replications, mean target truth = {truths.mean():.3f}")
print(f"{'estimator':<50}{'mean':>7}{'bias':>8}{'sd':>7}")
for k, v in est.items():
    v = np.array(v)
    print(f"{k:<50}{v.mean():7.3f}{(v - truths).mean():+8.3f}{v.std():7.3f}")

100 replications, mean target truth = 2.001
estimator                                            mean    bias     sd
source-adjusted, no transport                       1.501  -0.501  0.048
DR | selection right, outcome right                 1.996  -0.006  0.054
DR | selection WRONG (linear), outcome right        1.994  -0.007  0.092
DR | selection right, outcome WRONG                 2.006  +0.005  0.139
weighted-AIPW | selection WRONG, outcome right      2.099  +0.098  0.091
weighted-AIPW | selection right, outcome right      1.996  -0.005  0.054


## What this notebook shows
- A confounding-adjusted source estimate is still not the target effect when effect modifiers
  differ (sections 3 and 6), and the naive source estimate is off for two reasons at once:
  confounding and transport shift (section 3).
- A doubly robust transport estimator that evaluates the outcome model on the target
  recovers the target ATE on average (sections 4 and 6).
- The first-draft weighted-AIPW form depends on the selection model being right; the DR form
  is much less sensitive to it, which is what the "WRONG" rows test (section 6). Residual
  finite-sample bias with a wrong model is small compared with the spread, but not exactly zero.
- Overlap diagnostics (ESS, weight concentration, covariate ranges) indicate whether the
  correction is trustworthy (section 5).

## What it does not show
- Real data, or a real choice of target population
- Federated transportability (the selection model needs covariates from both source and
  target sites)
- Sensitivity to unmeasured effect modifiers
- Formal transportability assumptions (consistency, exchangeability over the population
  indicator conditional on X, positivity of selection) are assumed to hold by construction